# 03. 공급사 + 상품 데이터 전반 분석

01/02 크롤링 결과 두 파일을 각각 분석합니다.
`공급업체_분석.ipynb` / `공급업체_정보_분석.ipynb`와 목적은 비슷하지만, 그 두 노트북은
**입점 승인현황 엑셀(`공급업체_정보.xlsx`, 신청일/승인일 등 포함)**을 대상으로 하는
별개 데이터용이라 이 노트북과는 입력이 다릅니다.

- **입력**:
  - `supplier_detail_result.xlsx` (02 노트북 출력, 업체별 1행 — Part A. 업체분석에 사용)
  - `partner_goods_full.xlsx` (01 노트북 출력, 상품별 1행 — Part B. 상품분석에 사용)
  - `products_with_supplier_info.xlsx` (02 마지막 조인 출력 — 있으면 Part C. 교차분석에 추가로 사용, 없어도 실행됨)
- **출력**: 없음 (표/그래프로 화면 출력만 합니다)
- **구성**: Part A(업체분석) → Part B(상품분석) → Part C(선택, 교차분석) → 요약 인사이트


In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd

try:
    import koreanize_matplotlib  # noqa: F401
except Exception:
    plt.rcParams['axes.unicode_minus'] = False
    print("koreanize-matplotlib 미설치 → 한글이 깨지면 'pip install koreanize-matplotlib' 후 재실행하세요.")

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.dpi'] = 110

ACCENT = '#2563eb'
ACCENT2 = '#f59e0b'
MUTE = '#cbd5e1'
pd.set_option('display.max_columns', None)

# ============================================================
# 0. 데이터 로드
# ============================================================
DATA_DIR = "../data"
SUPPLIER_PATH = f"{DATA_DIR}/supplier_detail_result.xlsx"   # 02 노트북 출력 (업체별 1행)
GOODS_PATH = f"{DATA_DIR}/partner_goods_full.xlsx"        # 01 노트북 출력 (상품별 1행)
JOIN_PATH = f"{DATA_DIR}/products_with_supplier_info.xlsx"    # 02 노트북 마지막 조인 출력 (있으면 Part C에서 사용)

df_supplier = pd.read_excel(SUPPLIER_PATH, dtype=str).fillna("")
df_goods = pd.read_excel(GOODS_PATH, dtype=str).fillna("")

try:
    df_join = pd.read_excel(JOIN_PATH, dtype=str).fillna("")
    HAS_JOIN = True
except FileNotFoundError:
    df_join = None
    HAS_JOIN = False
    print(f"[안내] {JOIN_PATH}를 찾을 수 없어 Part C(교차분석)는 건너뜁니다. "
          f"02 노트북(또는 giftco_supplier_crawler.py)을 끝까지 실행하면 생성됩니다.")

print(f"공급사 데이터: {df_supplier.shape[0]:,}행 x {df_supplier.shape[1]}열")
print(f"상품 데이터  : {df_goods.shape[0]:,}행 x {df_goods.shape[1]}열")
if HAS_JOIN:
    print(f"조인 데이터  : {df_join.shape[0]:,}행 x {df_join.shape[1]}열")


## Part A. 업체(공급사) 분석 — `supplier_detail_result.xlsx` 기준


In [ ]:
# ============================================================
# A-1. 파생 컬럼 만들기
# ============================================================
def _biz_valid(x):
    """사업자번호 유효성: 숫자만 뽑아 10자리이고 전부 0이 아니면 유효."""
    d = re.sub(r"\D", "", str(x))
    return len(d) == 10 and set(d) != {"0"}

df_supplier["상품수_n"] = pd.to_numeric(df_supplier["상품수"], errors="coerce").fillna(0).astype(int)
df_supplier["사업자번호_유효"] = df_supplier["공급처_사업자번호"].apply(_biz_valid)
df_supplier["전화보유"] = df_supplier["전화번호"].str.len() > 0
df_supplier["휴대폰보유"] = df_supplier["휴대폰번호"].str.len() > 0
df_supplier["이메일보유"] = df_supplier["이메일"].str.contains("@", na=False)
df_supplier["주소보유"] = df_supplier["주소"].str.len() > 0

n_vendor = len(df_supplier)
print(f"총 공급업체 수: {n_vendor:,}개")


In [ ]:
# ============================================================
# A-2. 연락처 · 사업자정보 데이터 품질
# ============================================================
info_rate = {
    "전화번호 보유": df_supplier["전화보유"].mean(),
    "휴대폰 보유": df_supplier["휴대폰보유"].mean(),
    "이메일 보유": df_supplier["이메일보유"].mean(),
    "주소 보유": df_supplier["주소보유"].mean(),
    "사업자번호 유효": df_supplier["사업자번호_유효"].mean(),
    "대표자 정보 있음": (df_supplier["공급처_대표자"].str.len() > 0).mean(),
    "업태 정보 있음": (df_supplier["공급처_업태"].str.len() > 0).mean(),
}
rate = (pd.Series(info_rate) * 100).sort_values()

fig, ax = plt.subplots()
bars = ax.barh(rate.index, rate.values, color=ACCENT)
for b, v in zip(bars, rate.values):
    ax.text(v + 1, b.get_y() + b.get_height() / 2, f"{v:.1f}%", va="center", fontsize=9)
ax.set_title("항목별 데이터 보유율", fontsize=13, fontweight="bold")
ax.set_xlabel("보유율 (%)")
ax.set_xlim(0, 100)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# A-3. 크롤링 성공률 (상세페이지 / 발주페이지)
# ============================================================
status_counts = df_supplier["상태"].value_counts()
print("[상세페이지 크롤링 상태]")
print(status_counts)

fig, ax = plt.subplots()
status_counts.plot(kind="bar", ax=ax, color=ACCENT)
ax.set_title("상세페이지 크롤링 상태", fontsize=13, fontweight="bold")
ax.set_ylabel("업체 수")
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

if "발주_상태" in df_supplier.columns:
    order_status_counts = df_supplier["발주_상태"].value_counts()
    print("\n[발주페이지(사업자정보) 크롤링 상태]")
    print(order_status_counts)

    fig, ax = plt.subplots()
    order_status_counts.plot(kind="bar", ax=ax, color=ACCENT2)
    ax.set_title("발주페이지(사업자정보) 크롤링 상태", fontsize=13, fontweight="bold")
    ax.set_ylabel("업체 수")
    plt.xticks(rotation=20)
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# A-4. 업태 · 종목 분포 (상위 15개)
# ============================================================
for col, title in [("공급처_업태", "업태별 업체 수 (상위 15)"), ("공급처_종목", "종목별 업체 수 (상위 15)")]:
    top = df_supplier[df_supplier[col].str.len() > 0][col].value_counts().head(15).iloc[::-1]
    if len(top) == 0:
        print(f"[안내] {col} 데이터가 없어 건너뜁니다.")
        continue
    fig, ax = plt.subplots(figsize=(9, max(4, len(top) * 0.35)))
    ax.barh(top.index, top.values, color=ACCENT)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("업체 수")
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# A-5. 업체별 상품수 분포
# ============================================================
seg_bins = [-1, 0, 9, 49, 99, float("inf")]
seg_labels = ["0개", "1~9", "10~49", "50~99", "100개+"]
df_supplier["상품수구간"] = pd.cut(df_supplier["상품수_n"], bins=seg_bins, labels=seg_labels)

seg = df_supplier["상품수구간"].value_counts().reindex(seg_labels)
fig, ax = plt.subplots()
bars = ax.bar(seg.index, seg.values, color=ACCENT)
for b, v in zip(bars, seg.values):
    ax.text(b.get_x() + b.get_width() / 2, v + max(seg.values, default=0) * 0.01, f"{v}", ha="center", fontsize=9)
ax.set_title("업체별 상품수 구간 분포", fontsize=13, fontweight="bold")
ax.set_ylabel("업체 수")
plt.tight_layout(); plt.show()

top15_vendor = df_supplier.nlargest(15, "상품수_n")[["업체명", "상품수_n"]].iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top15_vendor["업체명"], top15_vendor["상품수_n"], color=ACCENT)
ax.set_title("상품수 상위 15개 업체", fontsize=13, fontweight="bold")
ax.set_xlabel("상품수")
plt.tight_layout(); plt.show()


## Part B. 상품 분석 — `partner_goods_full.xlsx` 기준


In [ ]:
# ============================================================
# B-1. 파생 컬럼 만들기
# ============================================================
def _to_number(s):
    """가격/수량 텍스트에서 숫자만 뽑아 float으로 (쉼표, 원, % 등 제거)."""
    s = re.sub(r"[^0-9.\-]", "", str(s))
    return pd.to_numeric(s, errors="coerce")

for col in ["공급가", "판매가1", "판매가7", "마진율", "기본수량"]:
    if col in df_goods.columns:
        df_goods[f"{col}_n"] = df_goods[col].apply(_to_number)

df_goods["등록일_d"] = pd.to_datetime(df_goods["최초등록일"], errors="coerce")

n_goods = len(df_goods)
print(f"총 상품 수: {n_goods:,}개")
print(f"업체 수(상품 파일 기준): {df_goods['업체명'].nunique():,}개")


In [ ]:
# ============================================================
# B-2. 카테고리 분포 (상위 15)
# ============================================================
cat_counts = df_goods[df_goods["카테고리"].str.len() > 0]["카테고리"].value_counts().head(15).iloc[::-1]
if len(cat_counts):
    fig, ax = plt.subplots(figsize=(9, max(4, len(cat_counts) * 0.35)))
    ax.barh(cat_counts.index, cat_counts.values, color=ACCENT)
    ax.set_title("상품 카테고리 분포 (상위 15)", fontsize=13, fontweight="bold")
    ax.set_xlabel("상품 수")
    plt.tight_layout(); plt.show()
else:
    print("[안내] 카테고리 데이터가 없어 건너뜁니다.")


In [ ]:
# ============================================================
# B-3. 가격 · 마진율 분포
# ============================================================
price_col = "판매가1_n"
prices = df_goods[price_col].dropna()
prices = prices[prices > 0]
if len(prices):
    print(f"[{price_col}] 평균: {prices.mean():,.0f}원 / 중앙값: {prices.median():,.0f}원 / "
          f"최소: {prices.min():,.0f}원 / 최대: {prices.max():,.0f}원")
    fig, ax = plt.subplots()
    ax.hist(prices.clip(upper=prices.quantile(0.99)), bins=40, color=ACCENT)
    ax.set_title("판매가1 분포 (상위 1% 제외)", fontsize=13, fontweight="bold")
    ax.set_xlabel("가격(원)")
    ax.set_ylabel("상품 수")
    plt.tight_layout(); plt.show()

margin_col = "마진율_n"
margins = df_goods[margin_col].dropna() if margin_col in df_goods.columns else pd.Series(dtype=float)
if len(margins):
    print(f"[마진율] 평균: {margins.mean():.1f} / 중앙값: {margins.median():.1f}")
    fig, ax = plt.subplots()
    ax.hist(margins.clip(lower=margins.quantile(0.01), upper=margins.quantile(0.99)), bins=40, color=ACCENT2)
    ax.set_title("마진율 분포 (상하위 1% 제외)", fontsize=13, fontweight="bold")
    ax.set_xlabel("마진율")
    ax.set_ylabel("상품 수")
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# B-4. 진열 · 과세 · 퍼가기금지 현황
# ============================================================
for col, title in [("진열", "진열 여부 분포"), ("과세", "과세 여부 분포"), ("퍼가기금지", "퍼가기금지 여부 분포")]:
    if col not in df_goods.columns:
        continue
    counts = df_goods[df_goods[col].str.len() > 0][col].value_counts()
    if len(counts) == 0:
        continue
    print(f"[{col}] {dict(counts)}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, ["진열", "과세", "퍼가기금지"]):
    if col in df_goods.columns:
        counts = df_goods[df_goods[col].str.len() > 0][col].value_counts()
        counts.plot(kind="bar", ax=ax, color=ACCENT)
        ax.set_title(col, fontsize=11, fontweight="bold")
        ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# B-5. 등록일 추이 (월별 상품 등록 건수)
# ============================================================
reg = df_goods.dropna(subset=["등록일_d"]).copy()
if len(reg):
    monthly = reg.groupby(reg["등록일_d"].dt.to_period("M")).size()
    monthly.index = monthly.index.to_timestamp()

    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.bar(monthly.index, monthly.values, width=20, color=ACCENT, alpha=0.85)
    ax.set_title("월별 상품 등록 건수", fontsize=13, fontweight="bold")
    ax.set_xlabel("등록 연월")
    ax.set_ylabel("등록 상품 수")
    plt.tight_layout(); plt.show()
else:
    print("[안내] 등록일 데이터를 날짜로 변환할 수 없어 건너뜁니다.")


In [ ]:
# ============================================================
# B-6. 상품 파일 기준 업체별 상품수 Top 15 (A-5와 교차 확인용)
# ============================================================
vendor_goods_count = df_goods.groupby("업체명").size().sort_values(ascending=False).head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(vendor_goods_count.index, vendor_goods_count.values, color=ACCENT2)
ax.set_title("상품 파일 기준 업체별 상품수 Top 15", fontsize=13, fontweight="bold")
ax.set_xlabel("상품수")
plt.tight_layout(); plt.show()

print("[참고] A-5의 'supplier_detail_result.xlsx' 상품수(크롤링 시점 build_targets 계산값)와 "
      "여기 B-6의 실제 상품 파일 집계가 같은 업체 기준으로 크게 다르면 두 파일 수집 시점이 어긋났을 수 있습니다.")


## Part C. (선택) 조인 파일 기반 교차분석 — `products_with_supplier_info.xlsx`


In [ ]:
# ============================================================
# C-1. 카테고리별 대표 업태 (조인 파일이 있을 때만)
# ============================================================
if HAS_JOIN and "카테고리" in df_join.columns and "공급처_업태" in df_join.columns:
    cross = (
        df_join[(df_join["카테고리"].str.len() > 0) & (df_join["공급처_업태"].str.len() > 0)]
        .groupby(["카테고리", "공급처_업태"])
        .size()
        .reset_index(name="건수")
    )
    top_biztype_per_category = (
        cross.sort_values("건수", ascending=False)
        .groupby("카테고리")
        .head(1)
        .sort_values("건수", ascending=False)
        .head(15)
    )
    print("[카테고리별 가장 많은 업태 Top 15]")
    top_biztype_per_category
else:
    print("[안내] 조인 파일이 없거나 필요한 컬럼이 없어 Part C를 건너뜁니다.")


In [ ]:
# ============================================================
# 요약 인사이트
# ============================================================
print("=" * 46)
print(" 공급사 + 상품 데이터 분석 요약")
print("=" * 46)
print(f"[업체] 총 공급업체 수      : {n_vendor:,}개")
print(f"[업체] 상세페이지 크롤링 성공 : {(df_supplier['상태'] == 'OK').sum():,}개 "
      f"({(df_supplier['상태'] == 'OK').mean() * 100:.1f}%)")
if "발주_상태" in df_supplier.columns:
    print(f"[업체] 사업자정보 크롤링 성공 : {(df_supplier['발주_상태'] == 'OK').sum():,}개 "
          f"({(df_supplier['발주_상태'] == 'OK').mean() * 100:.1f}%)")
print(f"[업체] 전화/휴대폰 보유     : {df_supplier['전화보유'].mean() * 100:.1f}% / "
      f"{df_supplier['휴대폰보유'].mean() * 100:.1f}%")
print(f"[업체] 사업자번호 유효      : {df_supplier['사업자번호_유효'].mean() * 100:.1f}%")
print("-" * 46)
print(f"[상품] 총 상품 수          : {n_goods:,}개")
print(f"[상품] 상품 파일 업체 수    : {df_goods['업체명'].nunique():,}개")
if "판매가1_n" in df_goods.columns and df_goods["판매가1_n"].notna().any():
    print(f"[상품] 판매가1 평균/중앙값  : {df_goods['판매가1_n'].mean():,.0f}원 / "
          f"{df_goods['판매가1_n'].median():,.0f}원")
if "카테고리" in df_goods.columns:
    top_cat = df_goods[df_goods["카테고리"].str.len() > 0]["카테고리"].value_counts()
    if len(top_cat):
        print(f"[상품] 최다 카테고리       : {top_cat.index[0]} ({top_cat.iloc[0]:,}개)")
print("=" * 46)
